In [1]:
from datetime import datetime, timedelta
from typing import Dict, Tuple
from Kobo.kobo_toolbox_client import KoboToolboxClient
import pandas as pd

# Initialize client
kobo_client = KoboToolboxClient(
	url="https://kf.kobotoolbox.org/",
	token="f7196ace51297457806144859ca0991c1747c11c"
)

try:
    # ===========================================
    # 1. FORMS/ASSETS ENDPOINTS
    # ===========================================
    
    # Get all forms
    print("\n=== Getting all forms ===")
    forms = kobo_client.get_all_forms()
    print(f"Found {len(forms)} forms")
    for form in forms[:3]:  # Print first 3 forms
        print(f" - {form.name} (UID: {form.uid})")
    
    if not forms:
        print("No forms found, exiting...")
        exit()
    
    # Get details of first form
    first_form_uid = forms[0].uid
    print("\n=== Getting form details ===")
    form_details = kobo_client.get_form_details(first_form_uid)
    print(f"Details for form '{forms[0].name}':")
    print(f" - Date created: {form_details.get('date_created')}")
    print(f" - Submission count: {form_details.get('deployment__submission_count')}")
    print(f" - Has repeats: {form_details.get('has_repeats', False)}")
    
    # ===========================================
    # 2. SUBMISSIONS ENDPOINTS
    # ===========================================
    
    # Get submissions (basic)
    print("\n=== Getting submissions ===")
    submissions= kobo_client.get_submissions(first_form_uid)

    assert isinstance(submissions, pd.DataFrame), "submissions must be a DataFrame"

    print(f"Retrieved {len(submissions)} submissions")
    print("First submission columns:", submissions.columns.tolist()[:5])  # Print first 5 columns
    
    # Get submissions with repeats (separate DataFrames)
    print("\n=== Getting submissions with repeats ===")
    main_data , repeat_data  = kobo_client.get_submissions(
        first_form_uid,
        include_repeats=True,
        join_repeats=False
    )
    
    assert isinstance(main_data, pd.DataFrame), "main_data must be a DataFrame"
    assert isinstance(repeat_data, dict), "repeat_data must be a Dict[str, pd.DataFrame]"
    
    print(f"Main data shape: {main_data.shape}")
    if repeat_data:
        print(f"Found {len(repeat_data)} repeat groups")
        for name, df in repeat_data.items():
            print(f" - {name}: {len(df)} records")
    
    # Get submissions with repeats joined
    print("\n=== Getting joined submissions ===")
    joined_data = kobo_client.get_submissions(
        first_form_uid,
        include_repeats=True,
        join_repeats=True
    )
    
    assert isinstance(joined_data, pd.DataFrame), "main_data must be a DataFrame"

    print(f"Joined data shape: {joined_data.shape}")
    print("Columns with repeats:", [c for c in joined_data.columns if '_' in c][:5])  # Sample repeat columns
    
    # Process submissions in batches (for large forms)
    print("\n=== Processing submissions in batches ===")
    total_records = 0
    for batch in kobo_client.get_all_submissions(first_form_uid, batch_size=500):
        total_records += len(batch)
        print(f"Processing batch with {len(batch)} records...")
        # Here you would do your actual processing
    print(f"Processed total of {total_records} records")
    
    # ===========================================
    # 3. DATA EXPORT METHODS
    # ===========================================
    
    # Export to CSV
    print("\n=== Exporting to CSV ===")
    csv_path = "form_data.csv"
    result = kobo_client.export_form_data(
        first_form_uid,
        output_format='csv',
        output_path=csv_path
    )
    print(f"Data exported to {result}")
    
    # Export to Excel
    print("\n=== Exporting to Excel ===")
    excel_path = "form_data.xlsx"
    result = kobo_client.export_form_data(
        first_form_uid,
        output_format='excel',
        output_path=excel_path
    )
    print(f"Data exported to {result}")
    
    # Get data as CSV string
    print("\n=== Getting data as CSV string ===")
    csv_data = kobo_client.export_form_data(
        first_form_uid,
        output_format='csv'
    )
    
    assert isinstance(csv_data, str), "csv_data must be a str"

    print(f"Got CSV data (first 100 chars): {csv_data[:100]}...")
    
    # ===========================================
    # 4. MEDIA METHODS (PLACEHOLDER)
    # ===========================================
    
    print("\n=== Media methods (placeholder) ===")
    media_list = kobo_client.get_form_media(first_form_uid)
    print(f"Media attachments: {len(media_list)}")
    
    # This would raise NotImplementedError
    try:
        if media_list:
            kobo_client.download_media(first_form_uid, media_list[0]['filename'])
    except NotImplementedError as e:
        print(f"Media download not implemented: {e}")

except Exception as e:
    print(f"\nError occurred: {e}")
finally:
    print("\n=== Script completed ===")


=== Getting all forms ===
Found 2 forms
 - TEST2 (UID: aECXzcrhZ59fbakYU8sPCu)
 - Test (UID: aW8JMa9NxJ3h2Y3jjkCSPz)

=== Getting form details ===
Details for form 'TEST2':
 - Date created: 2025-07-04T18:31:11.692169Z
 - Submission count: None
 - Has repeats: False

=== Getting submissions ===
Retrieved 2 submissions
First submission columns: ['start', 'end', 'Question_1', 'Question_2', 'What_happen_if_we_gr_Damascus_University']

=== Getting submissions with repeats ===

Error occurred: KoboToolboxClient.get_submissions() got an unexpected keyword argument 'include_repeats'

=== Script completed ===


In [3]:
import pandas as pd

formData = pd.read_csv('./form_data.csv')

formData.head()

,start,end,New_Question,New_Question_001,_id,__version__,_uuid,rootUuid,_attachments,_status,_submission_time,_tags,_notes,_validation_status,_submitted_by,_index
0,2025-05-05T19:25:58.662+03:00,2025-05-05T19:26:01.973+03:00,option_3,option_1,474432827,vQEroPJdBVoD4DswYfb7sQ,4f9a39fb-6a2e-4335-b349-e88d914fcc88,uuid:4f9a39fb-6a2e-4335-b349-e88d914fcc88,NaN,submitted_via_web,2025-05-05T16:26:02,NaN,NaN,NaN,NaN,1
1,2025-05-05T19:26:01.982+03:00,2025-05-05T19:26:06.374+03:00,test,option_2,474432877,vQEroPJdBVoD4DswYfb7sQ,9f379953-9d0d-4929-8cb1-1df2289b52fe,uuid:9f379953-9d0d-4929-8cb1-1df2289b52fe,NaN,submitted_via_web,2025-05-05T16:26:06,NaN,NaN,NaN,NaN,2
